# BuySense - Data Preprocessing and Feature Engineering

This notebook prepares the Online Shoppers Purchasing Intention Dataset for machine learning.

The objectives are to:

- separate features and target
- perform a stratified train-test split
- correctly identify numerical, categorical and Boolean variables
- handle categorical encoding
- apply scaling where required
- create meaningful engineered features
- prevent data leakage
- prepare reusable preprocessing pipelines for later model development

In [1]:
from google.colab import files

uploaded = files.upload()

Saving online_shoppers_intention.csv to online_shoppers_intention.csv


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", None)

In [3]:
df = pd.read_csv("online_shoppers_intention.csv")

In [4]:
df.shape

(12330, 18)

In [5]:
X = df.drop(columns=["Revenue"])
y = df["Revenue"].astype(int)

In [6]:
X.shape

(12330, 17)

In [7]:
y.value_counts()

,count
Revenue,
0,10422
1,1908


### Target Preparation

`Revenue` was converted from Boolean values to binary values:

- `False = 0` → No Purchase
- `True = 1` → Purchase

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [9]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (9864, 17)
X_test : (2466, 17)
y_train: (9864,)
y_test : (2466,)


In [10]:
print("Training distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training distribution:
Revenue
0    84.529603
1    15.470397
Name: proportion, dtype: float64

Testing distribution:
Revenue
0    84.509327
1    15.490673
Name: proportion, dtype: float64


### Train-Test Split

The dataset was divided into:

- 80% training data
- 20% testing data

A stratified split was used because `Revenue` is imbalanced. This preserves approximately the same purchase and non-purchase proportions in both datasets.

The test set will remain untouched until final model evaluation.

In [11]:
def add_engineered_features(data):
    data = data.copy()

    data["TotalPages"] = (
        data["Administrative"]
        + data["Informational"]
        + data["ProductRelated"]
    )

    data["TotalDuration"] = (
        data["Administrative_Duration"]
        + data["Informational_Duration"]
        + data["ProductRelated_Duration"]
    )

    data["AvgDurationPerPage"] = np.where(
        data["TotalPages"] > 0,
        data["TotalDuration"] / data["TotalPages"],
        0
    )

    data["ProductPageShare"] = np.where(
        data["TotalPages"] > 0,
        data["ProductRelated"] / data["TotalPages"],
        0
    )

    data["ProductDurationShare"] = np.where(
        data["TotalDuration"] > 0,
        data["ProductRelated_Duration"] / data["TotalDuration"],
        0
    )

    return data

In [12]:
X_train = add_engineered_features(X_train)
X_test = add_engineered_features(X_test)

In [13]:
X_train[
    [
        "TotalPages",
        "TotalDuration",
        "AvgDurationPerPage",
        "ProductPageShare",
        "ProductDurationShare"
    ]
].head()

,TotalPages,TotalDuration,AvgDurationPerPage,ProductPageShare,ProductDurationShare
4263,42,2078.005952,49.476332,0.809524,0.647619
5905,4,157.200000,39.300000,1.000000,1.000000
9434,4,42.000000,10.500000,1.000000,1.000000
3505,19,1543.566667,81.240351,0.894737,0.781027
2067,32,827.646212,25.863944,1.000000,1.000000


### Feature Engineering

The following behavioural features were created:

- `TotalPages` – total number of pages visited during the session
- `TotalDuration` – total time spent across all page types
- `AvgDurationPerPage` – average time spent per visited page
- `ProductPageShare` – proportion of visited pages that were product-related
- `ProductDurationShare` – proportion of browsing time spent on product-related pages

These features provide a more direct representation of overall engagement and product interest.

In [14]:
categorical_features = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType"
]

In [15]:
boolean_features = [
    "Weekend"
]

In [16]:
numeric_features = [
    col for col in X_train.columns
    if col not in categorical_features + boolean_features
]

In [17]:
numeric_features

['Administrative',
 'Administrative_Duration',
 'Informational',
 'Informational_Duration',
 'ProductRelated',
 'ProductRelated_Duration',
 'BounceRates',
 'ExitRates',
 'PageValues',
 'SpecialDay',
 'TotalPages',
 'TotalDuration',
 'AvgDurationPerPage',
 'ProductPageShare',
 'ProductDurationShare']

### Outlier Treatment Decision

The EDA identified many statistical outliers in page-count, duration and behavioural variables.

However, these values may represent valid high-engagement browsing sessions. Therefore, outliers are retained at this stage rather than removed automatically.

Transformations or alternative treatments may later be tested if model performance indicates they are necessary.

### Missing Value Treatment

No missing values were identified during data understanding and EDA.

Therefore, no imputation is required for the original dataset.

### Duplicate Handling Decision

A total of 201 records were identified as belonging to exact duplicate groups.

However, the dataset does not contain a unique session identifier. Therefore, it is not possible to confirm whether these rows are accidental duplicates of the same browsing session or separate sessions with identical recorded behaviour.

Since all duplicated observations belong to the non-purchase class, removing them could also change the class distribution.

For this reason, the duplicate records will be retained initially rather than removed automatically.

Later, a sensitivity analysis can compare model performance with and without exact duplicates to determine whether they have a meaningful effect on the results.

In [18]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [19]:
numeric_pipeline_scaled = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

In [20]:
preprocessor_scaled = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline_scaled,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [21]:
preprocessor_tree = ColumnTransformer(
    transformers=[
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [22]:
X_train_scaled = preprocessor_scaled.fit_transform(X_train)

In [23]:
X_train_scaled.shape

(9864, 79)

In [24]:
X_test_scaled = preprocessor_scaled.transform(X_test)

### Leakage Prevention

The preprocessing pipeline is fitted only on the training data.

The test data is transformed using the already-fitted preprocessing pipeline.

This prevents information from the test set from influencing the preprocessing stage.

In [26]:
X_train_without_pagevalues = X_train.drop(columns=["PageValues"])
X_test_without_pagevalues = X_test.drop(columns=["PageValues"])

### PageValues Leakage Strategy

EDA showed that `PageValues` is strongly associated with `Revenue`.

Because its availability at prediction time must be confirmed, later modelling will compare:

1. models including `PageValues`
2. models excluding `PageValues`

This will help determine whether model performance depends heavily on a potentially leakage-prone feature.

### Feature Selection Plan

No features are removed solely based on EDA.

During model development, feature importance, correlation, model coefficients and validation performance will be used to determine whether any predictors can be removed without reducing predictive performance.

Potential redundancy such as the relationship between `BounceRates` and `ExitRates` will be investigated further during modelling.

### Class Imbalance Strategy

The target variable is imbalanced, with purchases representing approximately 15.47% of sessions.

The initial modelling stage will use:

- stratified cross-validation
- precision
- recall
- F1-score
- ROC-AUC / PR-AUC
- confusion matrix

Class weighting will be tested before using synthetic resampling methods such as SMOTE.

If SMOTE is later used, it will only be applied to the training data or within cross-validation folds to avoid data leakage.

In [28]:
pd.set_option("display.max_colwidth", None)

In [29]:
preprocessing_summary = pd.DataFrame({
    "Issue / Requirement": [
        "Missing values",
        "Duplicates",
        "Categorical variables",
        "Numerical scaling",
        "Outliers",
        "Class imbalance",
        "Feature engineering",
        "PageValues leakage risk",
        "Train-test leakage"
    ],

    "Decision": [
        "No imputation required",
        "Retain initially and test sensitivity later",
        "Use One-Hot Encoding",
        "Scale for Logistic Regression and KNN",
        "Retain initially; investigate only if required",
        "Use stratification and test class weighting",
        "Create engagement-based features",
        "Compare models with and without PageValues",
        "Fit preprocessing only on training data"
    ],

    "Reason": [
        "No missing values were found",
        "Identical browsing sessions may be legitimate",
        "Category codes do not represent numerical magnitude",
        "These algorithms are sensitive to feature scale",
        "Extreme values may represent genuine browsing behaviour",
        "Purchase class is much smaller than non-purchase class",
        "Improves representation of customer engagement",
        "Strong relationship with Revenue may create unrealistic performance",
        "Prevents information from test data affecting training"
    ]
})

preprocessing_summary

,Issue / Requirement,Decision,Reason
0,Missing values,No imputation required,No missing values were found
1,Duplicates,Retain initially and test sensitivity later,Identical browsing sessions may be legitimate
2,Categorical variables,Use One-Hot Encoding,Category codes do not represent numerical magnitude
3,Numerical scaling,Scale for Logistic Regression and KNN,These algorithms are sensitive to feature scale
4,Outliers,Retain initially; investigate only if required,Extreme values may represent genuine browsing behaviour
5,Class imbalance,Use stratification and test class weighting,Purchase class is much smaller than non-purchase class
6,Feature engineering,Create engagement-based features,Improves representation of customer engagement
7,PageValues leakage risk,Compare models with and without PageValues,Strong relationship with Revenue may create unrealistic performance
8,Train-test leakage,Fit preprocessing only on training data,Prevents information from test data affecting training


## Final Preprocessing Conclusion

The dataset has now been prepared for the model-development stage.

The target was separated from the predictors, and an 80/20 stratified train-test split was created to preserve the Revenue class distribution.

Categorical features will be handled using One-Hot Encoding, while numerical scaling will be applied only for algorithms that require it. Several engagement-based features were created to capture overall browsing activity and product interest.

Outliers and duplicate records were retained because the EDA did not provide sufficient evidence that they were erroneous. The class imbalance will initially be handled using stratified validation and class weighting rather than immediately applying synthetic resampling.

A preprocessing pipeline was created so that all learned transformations are fitted only on training data, reducing the risk of data leakage.

`PageValues` will be evaluated separately during modelling by comparing model performance with and without the feature.